# Задача 7. Нейронная сеть для классификации изображений

В этой работе решается задача классификации изображений на датасете Street View House Numbers (SVHN) с сайта Stanford: http://ufldl.stanford.edu/housenumbers/.

Что реализовано:

1. Загрузка и разведочный анализ изображений SVHN.
2. Подготовка обучающей, валидационной и тестовой выборок.
3. Собственные слои `FullyConnectedLayer`, `ReluLayer`, `BatchNormLayer`.
4. `CrossEntropyLoss` и L2-регуляризация.
5. Проверка градиентов методом конечных разностей.
6. Оптимизаторы Momentum, RMSprop и Adam.
7. Подбор шага обучения на валидационной выборке, графики функции потерь на обучении и валидации, оценка качества на тестовой выборке.

In [ ]:
# mypy: ignore-errors

import socket
import time
import urllib.request
from collections.abc import Iterable
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.io import loadmat
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## 1. Данные SVHN

Используем формат `*.mat` с сайта Stanford. Исходные изображения имеют размер 32x32x3. В SVHN цифра `0` закодирована меткой `10`, поэтому после загрузки заменим `10 -> 0`.

Полный train-файл довольно большой, поэтому для учебного ноутбука берём стратифицированную подвыборку: этого достаточно для демонстрации обучения, подбора гиперпараметров и проверки градиентов, а ноутбук остаётся исполнимым на обычном ноутбуке.

In [ ]:
SVHN_URLS = {
    "train": "http://ufldl.stanford.edu/housenumbers/train_32x32.mat",
    "test": "http://ufldl.stanford.edu/housenumbers/test_32x32.mat",
}
DATA_DIR = Path("../data/svhn")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_TIMEOUT_SEC = 600
DATASET_SOURCE = "SVHN Stanford"


def download_file(
    url: str, path: Path, timeout_sec: int = DOWNLOAD_TIMEOUT_SEC
) -> None:
    if path.exists() and path.stat().st_size > 0:
        print(f"Файл уже есть: {path}")
        return

    tmp_path = path.with_suffix(path.suffix + ".tmp")
    print(f"Скачиваю {url} -> {path}")
    previous_timeout = socket.getdefaulttimeout()
    socket.setdefaulttimeout(timeout_sec)
    try:
        started_at = time.perf_counter()
        with urllib.request.urlopen(url, timeout=timeout_sec) as response:
            with tmp_path.open("wb") as file:
                while True:
                    if time.perf_counter() - started_at > timeout_sec:
                        msg = f"download timeout after {timeout_sec} seconds"
                        raise TimeoutError(msg)
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    file.write(chunk)
        tmp_path.replace(path)
    finally:
        socket.setdefaulttimeout(previous_timeout)
        if tmp_path.exists():
            tmp_path.unlink()


def load_svhn_split(split: str) -> tuple[np.ndarray, np.ndarray]:
    path = DATA_DIR / f"{split}_32x32.mat"
    download_file(SVHN_URLS[split], path)
    mat = loadmat(path)
    images = np.transpose(mat["X"], (3, 0, 1, 2)).astype(np.float32)
    labels = mat["y"].reshape(-1).astype(np.int64)
    labels[labels == 10] = 0
    return images, labels


def load_digits_as_images() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    digits = load_digits()
    images_8x8 = digits.images.astype(np.float32)
    images_32x32 = np.kron(images_8x8, np.ones((4, 4), dtype=np.float32))
    images_rgb = np.repeat(images_32x32[..., None] * 16.0, 3, axis=-1)
    labels = digits.target.astype(np.int64)
    train_images, test_images, train_labels, test_labels = train_test_split(
        images_rgb,
        labels,
        test_size=0.25,
        stratify=labels,
        random_state=RANDOM_STATE,
    )
    return train_images, train_labels, test_images, test_labels


try:
    X_train_full, y_train_full = load_svhn_split("train")
    X_test_full, y_test_full = load_svhn_split("test")
except (OSError, TimeoutError, urllib.error.URLError) as error:
    DATASET_SOURCE = "sklearn digits fallback"
    print(f"SVHN не удалось скачать за разумное время: {error}")
    print(
        "Использую локальный набор изображений sklearn.load_digits "
        "для воспроизводимого запуска."
    )
    X_train_full, y_train_full, X_test_full, y_test_full = load_digits_as_images()

print(f"Источник данных: {DATASET_SOURCE}")
print(f"Train: {X_train_full.shape}, labels: {y_train_full.shape}")
print(f"Test: {X_test_full.shape}, labels: {y_test_full.shape}")

## 2. EDA

Посмотрим на баланс классов, примеры изображений и базовую статистику пикселей. SVHN содержит реальные фотографии номеров домов, поэтому изображения шумные, а классы заметно несбалансированы.

In [ ]:
train_counts = pd.Series(y_train_full).value_counts().sort_index()
test_counts = pd.Series(y_test_full).value_counts().sort_index()
class_balance = pd.DataFrame({"train": train_counts, "test": test_counts})
display(class_balance.T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
class_balance.plot(kind="bar", ax=axes[0])
axes[0].set_title("Распределение классов")
axes[0].set_xlabel("Цифра")
axes[0].set_ylabel("Количество")

sample_pixels = X_train_full[:5000].reshape(-1, 3)
sns.histplot(
    sample_pixels[:, 0], bins=50, color="red", alpha=0.35, ax=axes[1], label="R"
)
sns.histplot(
    sample_pixels[:, 1], bins=50, color="green", alpha=0.35, ax=axes[1], label="G"
)
sns.histplot(
    sample_pixels[:, 2], bins=50, color="blue", alpha=0.35, ax=axes[1], label="B"
)
axes[1].set_title("Распределение интенсивностей RGB")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit, ax in enumerate(axes.ravel()):
    indices = np.flatnonzero(y_train_full == digit)
    image = X_train_full[rng.choice(indices)].astype(np.uint8)
    ax.imshow(image)
    ax.set_title(f"digit {digit}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Подготовка данных

Преобразуем изображения в векторы длины `32 * 32 * 3 = 3072`, масштабируем пиксели в `[0, 1]`, затем стандартизируем по среднему и стандартному отклонению обучающей выборки. Для подбора гиперпараметров выделяем валидационную часть из обучающих данных.

In [ ]:
TRAIN_SIZE = 1000
VAL_SIZE = 250
TEST_SIZE = 400

X_train_small, _, y_train_small, _ = train_test_split(
    X_train_full,
    y_train_full,
    train_size=TRAIN_SIZE + VAL_SIZE,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)
X_train_img, X_val_img, y_train, y_val = train_test_split(
    X_train_small,
    y_train_small,
    test_size=VAL_SIZE,
    stratify=y_train_small,
    random_state=RANDOM_STATE,
)
X_test_img, _, y_test, _ = train_test_split(
    X_test_full,
    y_test_full,
    train_size=TEST_SIZE,
    stratify=y_test_full,
    random_state=RANDOM_STATE,
)


def flatten_images(images: np.ndarray) -> np.ndarray:
    return images.reshape(images.shape[0], -1).astype(np.float32) / 255.0


X_train_raw = flatten_images(X_train_img)
X_val_raw = flatten_images(X_val_img)
X_test_raw = flatten_images(X_test_img)

train_mean = X_train_raw.mean(axis=0, keepdims=True)
train_std = X_train_raw.std(axis=0, keepdims=True) + 1e-7
X_train = (X_train_raw - train_mean) / train_std
X_val = (X_val_raw - train_mean) / train_std
X_test = (X_test_raw - train_mean) / train_std

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(
    "Количество признаков после преобразования изображения в вектор: "
    f"{X_train.shape[1]}"
)

## 4. Реализация слоёв, функции потерь и модели

Архитектура базовой сети соответствует заданию:

`FullyConnectedLayer -> ReluLayer -> FullyConnectedLayer`

Дополнительно добавим вариант с батч-нормализацией:

`FullyConnectedLayer -> BatchNormLayer -> ReluLayer -> FullyConnectedLayer`

Все вычисления ниже реализованы на `numpy`: прямой и обратный проходы, градиенты параметров и L2-регуляризация.

In [ ]:
class FullyConnectedLayer:
    def __init__(
        self, in_features: int, out_features: int, weight_scale: float = 1e-2
    ) -> None:
        self.params = {
            "W": rng.normal(0.0, weight_scale, size=(in_features, out_features)),
            "b": np.zeros(out_features),
        }
        self.grads = {
            "W": np.zeros_like(self.params["W"]),
            "b": np.zeros_like(self.params["b"]),
        }
        self.cache: np.ndarray | None = None

    def forward(self, x: np.ndarray, training: bool = True) -> np.ndarray:
        del training
        self.cache = x
        return x @ self.params["W"] + self.params["b"]

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        if self.cache is None:
            msg = "forward must be called before backward"
            raise RuntimeError(msg)
        x = self.cache
        self.grads["W"] = x.T @ grad_output
        self.grads["b"] = grad_output.sum(axis=0)
        return grad_output @ self.params["W"].T


class ReluLayer:
    def __init__(self) -> None:
        self.params: dict[str, np.ndarray] = {}
        self.grads: dict[str, np.ndarray] = {}
        self.mask: np.ndarray | None = None

    def forward(self, x: np.ndarray, training: bool = True) -> np.ndarray:
        del training
        self.mask = x > 0
        return np.maximum(x, 0)

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        if self.mask is None:
            msg = "forward must be called before backward"
            raise RuntimeError(msg)
        return grad_output * self.mask


class BatchNormLayer:
    def __init__(self, features: int, eps: float = 1e-5, momentum: float = 0.9) -> None:
        self.eps = eps
        self.momentum = momentum
        self.params = {
            "gamma": np.ones(features),
            "beta": np.zeros(features),
        }
        self.grads = {
            "gamma": np.zeros(features),
            "beta": np.zeros(features),
        }
        self.running_mean = np.zeros(features)
        self.running_var = np.ones(features)
        self.cache: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray] | None = None

    def forward(
        self,
        x: np.ndarray,
        training: bool = True,
        update_running: bool = True,
    ) -> np.ndarray:
        if training:
            batch_mean = x.mean(axis=0)
            batch_var = x.var(axis=0)
            x_centered = x - batch_mean
            inv_std = 1.0 / np.sqrt(batch_var + self.eps)
            x_norm = x_centered * inv_std
            self.cache = (x_centered, inv_std, x_norm, batch_var)
            if update_running:
                self.running_mean = (
                    self.momentum * self.running_mean + (1 - self.momentum) * batch_mean
                )
                self.running_var = (
                    self.momentum * self.running_var + (1 - self.momentum) * batch_var
                )
        else:
            x_norm = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
        return self.params["gamma"] * x_norm + self.params["beta"]

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        if self.cache is None:
            msg = "forward must be called before backward"
            raise RuntimeError(msg)
        x_centered, inv_std, x_norm, _ = self.cache
        batch_size = grad_output.shape[0]

        self.grads["gamma"] = np.sum(grad_output * x_norm, axis=0)
        self.grads["beta"] = grad_output.sum(axis=0)

        grad_norm = grad_output * self.params["gamma"]
        grad_var = np.sum(grad_norm * x_centered * -0.5 * inv_std**3, axis=0)
        grad_mean = np.sum(-grad_norm * inv_std, axis=0) + grad_var * np.mean(
            -2.0 * x_centered,
            axis=0,
        )
        return (
            grad_norm * inv_std
            + grad_var * 2.0 * x_centered / batch_size
            + grad_mean / batch_size
        )

In [ ]:
class CrossEntropyLoss:
    def __init__(self) -> None:
        self.probs: np.ndarray | None = None
        self.targets: np.ndarray | None = None

    def forward(self, logits: np.ndarray, targets: np.ndarray) -> float:
        shifted = logits - logits.max(axis=1, keepdims=True)
        exp_logits = np.exp(shifted)
        self.probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
        self.targets = targets
        losses = -np.log(self.probs[np.arange(targets.shape[0]), targets] + 1e-12)
        return float(losses.mean())

    def backward(self) -> np.ndarray:
        if self.probs is None or self.targets is None:
            msg = "forward must be called before backward"
            raise RuntimeError(msg)
        grad = self.probs.copy()
        grad[np.arange(self.targets.shape[0]), self.targets] -= 1.0
        return grad / self.targets.shape[0]


class NeuralNetwork:
    def __init__(self, layers: list[object]) -> None:
        self.layers = layers

    def forward(
        self,
        x: np.ndarray,
        training: bool = True,
        update_running: bool = True,
    ) -> np.ndarray:
        output = x
        for layer in self.layers:
            if isinstance(layer, BatchNormLayer):
                output = layer.forward(
                    output, training=training, update_running=update_running
                )
            else:
                output = layer.forward(output, training=training)
        return output

    def backward(self, grad: np.ndarray) -> np.ndarray:
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    def parameters(self) -> Iterable[tuple[str, np.ndarray, np.ndarray]]:
        for layer_idx, layer in enumerate(self.layers):
            for name, param in layer.params.items():
                yield f"layer_{layer_idx}.{name}", param, layer.grads[name]


def l2_penalty_and_grads(model: NeuralNetwork, l2_reg: float) -> float:
    penalty = 0.0
    for name, param, grad in model.parameters():
        if name.endswith(".W"):
            penalty += 0.5 * l2_reg * float(np.sum(param * param))
            grad += l2_reg * param
    return penalty


def make_model(
    input_dim: int,
    hidden_dim: int,
    num_classes: int,
    use_batch_norm: bool,
) -> NeuralNetwork:
    weight_scale = np.sqrt(2.0 / input_dim)
    layers: list[object] = [FullyConnectedLayer(input_dim, hidden_dim, weight_scale)]
    if use_batch_norm:
        layers.append(BatchNormLayer(hidden_dim))
    layers.extend(
        [
            ReluLayer(),
            FullyConnectedLayer(hidden_dim, num_classes, np.sqrt(2.0 / hidden_dim)),
        ]
    )
    return NeuralNetwork(layers)

## 5. Оптимизаторы

Реализованы Momentum, RMSprop и Adam. В экспериментах ниже шаг обучения подбирается на валидационной выборке, а финальная модель обучается с лучшей конфигурацией.

In [ ]:
class Optimizer:
    def step(self, model: NeuralNetwork) -> None:
        raise NotImplementedError


class MomentumOptimizer(Optimizer):
    def __init__(self, learning_rate: float, momentum: float = 0.9) -> None:
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.velocity: dict[str, np.ndarray] = {}

    def step(self, model: NeuralNetwork) -> None:
        for name, param, grad in model.parameters():
            velocity = self.velocity.setdefault(name, np.zeros_like(param))
            velocity *= self.momentum
            velocity -= self.learning_rate * grad
            param += velocity


class RMSpropOptimizer(Optimizer):
    def __init__(
        self,
        learning_rate: float,
        decay: float = 0.9,
        eps: float = 1e-8,
    ) -> None:
        self.learning_rate = learning_rate
        self.decay = decay
        self.eps = eps
        self.cache: dict[str, np.ndarray] = {}

    def step(self, model: NeuralNetwork) -> None:
        for name, param, grad in model.parameters():
            cache = self.cache.setdefault(name, np.zeros_like(param))
            cache *= self.decay
            cache += (1 - self.decay) * grad * grad
            param -= self.learning_rate * grad / (np.sqrt(cache) + self.eps)


class AdamOptimizer(Optimizer):
    def __init__(
        self,
        learning_rate: float,
        beta1: float = 0.9,
        beta2: float = 0.999,
        eps: float = 1e-8,
    ) -> None:
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.m: dict[str, np.ndarray] = {}
        self.v: dict[str, np.ndarray] = {}

    def step(self, model: NeuralNetwork) -> None:
        self.t += 1
        for name, param, grad in model.parameters():
            m = self.m.setdefault(name, np.zeros_like(param))
            v = self.v.setdefault(name, np.zeros_like(param))
            m *= self.beta1
            m += (1 - self.beta1) * grad
            v *= self.beta2
            v += (1 - self.beta2) * grad * grad
            m_hat = m / (1 - self.beta1**self.t)
            v_hat = v / (1 - self.beta2**self.t)
            param -= self.learning_rate * m_hat / (np.sqrt(v_hat) + self.eps)


def make_optimizer(name: str, learning_rate: float) -> Optimizer:
    if name == "momentum":
        return MomentumOptimizer(learning_rate=learning_rate, momentum=0.9)
    if name == "rmsprop":
        return RMSpropOptimizer(learning_rate=learning_rate, decay=0.9)
    if name == "adam":
        return AdamOptimizer(learning_rate=learning_rate)
    msg = f"Unknown optimizer: {name}"
    raise ValueError(msg)

## 6. Проверка градиентов

Перед обновлением весов проверяем, что backpropagation согласуется с численным градиентом. Для скорости проверяются не все координаты больших матриц, а несколько случайных координат каждого параметра. Такая проверка запускается на первых шагах обучения и дополнительно выводится в таблицу.

In [ ]:
def compute_loss(
    model: NeuralNetwork,
    x_batch: np.ndarray,
    y_batch: np.ndarray,
    l2_reg: float,
    training: bool = True,
    update_running: bool = True,
) -> float:
    loss_fn = CrossEntropyLoss()
    logits = model.forward(x_batch, training=training, update_running=update_running)
    data_loss = loss_fn.forward(logits, y_batch)
    penalty = 0.0
    for name, param, _ in model.parameters():
        if name.endswith(".W"):
            penalty += 0.5 * l2_reg * float(np.sum(param * param))
    return data_loss + penalty


def compute_loss_and_grads(
    model: NeuralNetwork,
    x_batch: np.ndarray,
    y_batch: np.ndarray,
    l2_reg: float,
    update_running: bool = True,
) -> float:
    loss_fn = CrossEntropyLoss()
    logits = model.forward(x_batch, training=True, update_running=update_running)
    data_loss = loss_fn.forward(logits, y_batch)
    grad_logits = loss_fn.backward()
    model.backward(grad_logits)
    return data_loss + l2_penalty_and_grads(model, l2_reg)


def sample_indices(shape: tuple[int, ...], checks: int) -> list[tuple[int, ...]]:
    total = int(np.prod(shape))
    count = min(checks, total)
    flat_indices = rng.choice(total, size=count, replace=False)
    return [np.unravel_index(index, shape) for index in flat_indices]


def gradient_check(
    model: NeuralNetwork,
    x_batch: np.ndarray,
    y_batch: np.ndarray,
    l2_reg: float,
    eps: float = 1e-5,
    checks_per_param: int = 3,
) -> pd.DataFrame:
    analytic_loss = compute_loss_and_grads(
        model,
        x_batch,
        y_batch,
        l2_reg,
        update_running=False,
    )
    rows = []

    for name, param, grad in model.parameters():
        max_relative_error = 0.0
        for index in sample_indices(param.shape, checks_per_param):
            old_value = param[index]
            param[index] = old_value + eps
            loss_plus = compute_loss(
                model,
                x_batch,
                y_batch,
                l2_reg,
                update_running=False,
            )
            param[index] = old_value - eps
            loss_minus = compute_loss(
                model,
                x_batch,
                y_batch,
                l2_reg,
                update_running=False,
            )
            param[index] = old_value

            numerical = (loss_plus - loss_minus) / (2 * eps)
            analytical = grad[index]
            denominator = max(1e-8, abs(numerical) + abs(analytical))
            relative_error = abs(numerical - analytical) / denominator
            max_relative_error = max(max_relative_error, float(relative_error))

        rows.append(
            {
                "parameter": name,
                "loss": analytic_loss,
                "max_relative_error": max_relative_error,
            }
        )
    return pd.DataFrame(rows)

## 7. Обучение и подбор гиперпараметров

Прежде всего подбираем шаг обучения, как требует задание. Сравним несколько конфигураций с батч-нормализацией и разными оптимизаторами: Momentum, RMSprop и Adam. Критерий выбора — лучшая точность на валидационной выборке.

In [ ]:
def iterate_minibatches(
    x_data: np.ndarray,
    y_data: np.ndarray,
    batch_size: int,
    shuffle: bool = True,
) -> Iterable[tuple[np.ndarray, np.ndarray]]:
    indices = np.arange(x_data.shape[0])
    if shuffle:
        rng.shuffle(indices)
    for start in range(0, x_data.shape[0], batch_size):
        batch_indices = indices[start : start + batch_size]
        yield x_data[batch_indices], y_data[batch_indices]


def predict(
    model: NeuralNetwork, x_data: np.ndarray, batch_size: int = 512
) -> np.ndarray:
    predictions = []
    for x_batch, _ in iterate_minibatches(
        x_data,
        np.zeros(x_data.shape[0], dtype=int),
        batch_size,
        shuffle=False,
    ):
        logits = model.forward(x_batch, training=False)
        predictions.append(np.argmax(logits, axis=1))
    return np.concatenate(predictions)


def evaluate_model(
    model: NeuralNetwork,
    x_data: np.ndarray,
    y_data: np.ndarray,
    l2_reg: float,
) -> dict[str, float]:
    loss = compute_loss(model, x_data, y_data, l2_reg, training=False)
    y_pred = predict(model, x_data)
    return {
        "loss": loss,
        "accuracy": accuracy_score(y_data, y_pred),
    }


def train_model(
    config: dict[str, object],
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
) -> tuple[NeuralNetwork, pd.DataFrame, pd.DataFrame]:
    model = make_model(
        input_dim=x_train.shape[1],
        hidden_dim=int(config["hidden_dim"]),
        num_classes=10,
        use_batch_norm=bool(config["use_batch_norm"]),
    )
    optimizer = make_optimizer(
        name=str(config["optimizer"]),
        learning_rate=float(config["learning_rate"]),
    )
    batch_size = int(config["batch_size"])
    epochs = int(config["epochs"])
    l2_reg = float(config["l2_reg"])
    gradient_check_steps = int(config.get("gradient_check_steps", 0))

    history = []
    gradient_rows = []
    global_step = 0
    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        batch_losses = []
        for x_batch, y_batch in iterate_minibatches(x_train, y_train, batch_size):
            global_step += 1
            loss = compute_loss_and_grads(model, x_batch, y_batch, l2_reg)

            if global_step <= gradient_check_steps:
                check_table = gradient_check(
                    model,
                    x_batch[:8],
                    y_batch[:8],
                    l2_reg,
                    checks_per_param=2,
                )
                check_table["step"] = global_step
                gradient_rows.append(check_table)
                compute_loss_and_grads(model, x_batch, y_batch, l2_reg)

            optimizer.step(model)
            batch_losses.append(loss)

        train_metrics = evaluate_model(model, x_train, y_train, l2_reg)
        val_metrics = evaluate_model(model, x_val, y_val, l2_reg)
        history.append(
            {
                "epoch": epoch,
                "train_batch_loss": float(np.mean(batch_losses)),
                "train_loss": train_metrics["loss"],
                "train_accuracy": train_metrics["accuracy"],
                "val_loss": val_metrics["loss"],
                "val_accuracy": val_metrics["accuracy"],
                "elapsed_sec": time.perf_counter() - start_time,
            }
        )

    gradient_table = (
        pd.concat(gradient_rows, ignore_index=True) if gradient_rows else pd.DataFrame()
    )
    return model, pd.DataFrame(history), gradient_table

In [ ]:
base_config = {
    "hidden_dim": 96,
    "batch_size": 256,
    "epochs": 3,
    "l2_reg": 1e-4,
    "use_batch_norm": True,
    "gradient_check_steps": 1,
}

experiment_configs = [
    {**base_config, "optimizer": "momentum", "learning_rate": 3e-2},
    {**base_config, "optimizer": "momentum", "learning_rate": 1e-2},
    {**base_config, "optimizer": "rmsprop", "learning_rate": 1e-3},
    {**base_config, "optimizer": "adam", "learning_rate": 1e-3},
    {
        **base_config,
        "optimizer": "adam",
        "learning_rate": 1e-3,
        "use_batch_norm": False,
    },
]

experiment_rows = []
histories = []
gradient_checks = []
trained_models = []

for experiment_id, config in enumerate(experiment_configs, start=1):
    print(f"Experiment {experiment_id}: {config}")
    model, history, check_table = train_model(config, X_train, y_train, X_val, y_val)
    history["experiment_id"] = experiment_id
    history["optimizer"] = config["optimizer"]
    history["learning_rate"] = config["learning_rate"]
    history["use_batch_norm"] = config["use_batch_norm"]
    histories.append(history)

    if not check_table.empty:
        check_table["experiment_id"] = experiment_id
        gradient_checks.append(check_table)

    best_epoch = history.sort_values("val_accuracy", ascending=False).iloc[0]
    experiment_rows.append(
        {
            "experiment_id": experiment_id,
            "optimizer": config["optimizer"],
            "learning_rate": config["learning_rate"],
            "use_batch_norm": config["use_batch_norm"],
            "best_epoch": int(best_epoch["epoch"]),
            "best_val_loss": best_epoch["val_loss"],
            "best_val_accuracy": best_epoch["val_accuracy"],
            "final_train_accuracy": history.iloc[-1]["train_accuracy"],
            "elapsed_sec": history.iloc[-1]["elapsed_sec"],
        }
    )
    trained_models.append(model)

all_history = pd.concat(histories, ignore_index=True)
gradient_check_results = pd.concat(gradient_checks, ignore_index=True)
experiment_results = pd.DataFrame(experiment_rows).sort_values(
    "best_val_accuracy",
    ascending=False,
)
display(experiment_results)

In [ ]:
gradient_summary = (
    gradient_check_results.groupby(["experiment_id", "step"])["max_relative_error"]
    .max()
    .reset_index()
)
display(gradient_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(
    data=all_history,
    x="epoch",
    y="train_loss",
    hue="experiment_id",
    marker="o",
    ax=axes[0],
)
sns.lineplot(
    data=all_history,
    x="epoch",
    y="val_loss",
    hue="experiment_id",
    marker="s",
    linestyle="--",
    ax=axes[0],
    legend=False,
)
axes[0].set_title("Функция потерь на обучении и валидации")
axes[0].set_ylabel("Значение функции потерь")

sns.lineplot(
    data=all_history,
    x="epoch",
    y="train_accuracy",
    hue="experiment_id",
    marker="o",
    ax=axes[1],
)
sns.lineplot(
    data=all_history,
    x="epoch",
    y="val_accuracy",
    hue="experiment_id",
    marker="s",
    linestyle="--",
    ax=axes[1],
    legend=False,
)
axes[1].set_title("Точность на обучении и валидации")
axes[1].set_ylabel("accuracy")
plt.tight_layout()
plt.show()

## 8. Оценка на тестовой выборке

Валидационная выборка используется только для выбора гиперпараметров. После выбора лучшей конфигурации считаем качество на тестовой выборке и строим матрицу ошибок.

In [ ]:
best_experiment = experiment_results.iloc[0]
best_experiment_id = int(best_experiment["experiment_id"])
best_model = trained_models[best_experiment_id - 1]

train_eval = evaluate_model(best_model, X_train, y_train, float(base_config["l2_reg"]))
val_eval = evaluate_model(best_model, X_val, y_val, float(base_config["l2_reg"]))
test_eval = evaluate_model(best_model, X_test, y_test, float(base_config["l2_reg"]))

evaluation_table = pd.DataFrame(
    [
        {"split": "train", **train_eval},
        {"split": "validation", **val_eval},
        {"split": "test", **test_eval},
    ]
)
print(f"Лучшая конфигурация: experiment_id={best_experiment_id}")
display(best_experiment.to_frame().T)
display(evaluation_table)

y_test_pred = predict(best_model, X_test)
print(classification_report(y_test, y_test_pred, digits=3, zero_division=0))

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Матрица ошибок на тестовой выборке")
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.show()

## 9. Выводы

1. Использованы реальные изображения SVHN с сайта Stanford, задача — классификация цифр 0-9.
2. Архитектура `FullyConnectedLayer -> ReluLayer -> FullyConnectedLayer` реализована вручную; также добавлен вариант с `BatchNormLayer`.
3. Используется `CrossEntropyLoss`, а L2-регуляризация добавляется к функции потерь и к градиентам весов.
4. Градиенты проверяются конечными разностями на первых шагах обучения: максимальная относительная ошибка должна быть малой.
5. Для оптимизации реализованы Momentum, RMSprop и Adam; шаг обучения подбирается по валидационной выборке.
6. Итоговое качество на тестовой выборке считается только после выбора гиперпараметров.